# Objective is to use level set function on a simply supported beam

In [29]:
using Gridap
using GridapGmsh
using GridapTopOpt
using GridapEmbedded

In [ ]:
# # Call Model
# model_name = "SSB(100x25x25).msh"
# model_file = joinpath(@__DIR__,"..", "..", "Model_Creation", "Gmsh_Model", "Model", model_name)
# isfile(model_file) || error("Path does not exist: $model_file")

In [ ]:
# model = GmshDiscreteModel(model_file)

In [30]:
# Domain Size
xmax, ymax, zmax = (100.0, 10.0, 10.0)                 
dom = (0, xmax, 0, ymax, 0, zmax)   

# Mesh Partition Size
el_size = (500, 50, 50)                          

# Model Creation
model = CartesianDiscreteModel(dom, el_size);

In [ ]:
# Check For Meshing
result_path = joinpath(@__DIR__, "..", "..", "..", "Result", "Output","SSB(100x10x10)")
isdir(result_path) || mkpath(result_path)

writevtk(model, joinpath(result_path, "model_check"))

In [ ]:
# Boolean Valued indicator functions

# Boundary Conditions
prop_Γ_N = 0.2                          
prop_Γ_D = 0.2  

# Γ_N Neumann Boundary Conditions
f_Γ_N(x) = (
    x[1] ≈ xmax && 
    ymax/2-ymax*prop_Γ_N/2 - eps() <= x[2] <= ymax/2+ymax*prop_Γ_N/2 + eps()
)
# Γ_D Drichlet Boundary Conditions
f_Γ_D(x) = (
    x[1] ≈ 0.0 && 
    (x[2] <= ymax*prop_Γ_D + eps() || x[2] >= ymax-ymax*prop_Γ_D/2 - eps())
);

In [27]:
# Initial level set function
lsf_func = initial_lsf(4, 0.2)                  

#325 (generic function with 1 method)

In [ ]:
# Ω_bg = Triangulation(model)
hmin = minimum(get_element_diameters(model))

GenericCellField():
 num_cells: 2558
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 7666302487562048264

In [ ]:
g = VectorValue(0.0,1000.0/(250*200),0.0) ## force
const E = 25000.0
const ν = 0.2
const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = E/(2*(1+ν))
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε

In [19]:
order = 1
# load in Tag = 'D' (y=-10KN)
tol = ((hmin/(5000*order^2)))
max_steps = ceil(Int, order*100/hmin) 

20

In [ ]:
# Finite Difference Parameters

# Courant-Friedrichs-Lewy (CFL) number OR "Time step coefficient" for Hamilton-Jaccobi equation 
γ = 0.1                                             

 # Reininitialization equation
γ_reinit = 0.5                                     


# "max_steps" and "tol" is scaled by mesh size

# Max steps for advection
max_steps =  100

# Reininitialization tolerance
tol = 1e-4


# max_steps = floor(Int, order*minimum(hmin)/10)  

# tol = 1/ (5order^2) /minimum(hmin); 

0.0001

In [20]:
Ω = Triangulation(model)
Γ_N = BoundaryTriangulation(model, tags="D")
dΩ = Measure(Ω, 2*order)
dΓ_N = Measure(Γ_N, 2*order);

In [25]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
reffe_scalar = ReferenceFE(lagrangian,Float64,order)

V = TestFESpace(model,reffe;dirichlet_tags=["Hinge"])

d1 = VectorValue(0.0,0.0,0.0)
U = TrialFESpace(V,[d1])

V_φ = TestFESpace(model,reffe_scalar)
V_reg = TestFESpace(model,reffe_scalar;dirichlet_tags=["Roller"])
U_reg = TrialFESpace(V_reg,0);

In [28]:
φh = interpolate(lsf_func,V_φ)
interp = SmoothErsatzMaterialInterpolation(η = 2*maximum(get_el_Δ(model)))    # η = 2 ×  maximum side length of an element.
I,H,DH,ρ = interp.I,interp.H,interp.DH,interp.ρ;

MethodError: MethodError: no method matching get_cartesian_element_sizes(::Gridap.Geometry.UnstructuredDiscreteModel{3, 3, Float64, Gridap.Geometry.Oriented})
The function `get_cartesian_element_sizes` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  get_cartesian_element_sizes(!Matched::GridapDistributed.DistributedDiscreteModel)
   @ GridapTopOpt C:\Users\IIT BBSR\.julia\packages\GridapTopOpt\zQxvu\src\Utilities.jl:186
  get_cartesian_element_sizes(!Matched::CartesianDiscreteModel)
   @ GridapTopOpt C:\Users\IIT BBSR\.julia\packages\GridapTopOpt\zQxvu\src\Utilities.jl:181
